<img src="https://www.unad.edu.co/images/footer/logo-unad-acreditacion-min.png" width="780" height="140" align="right"/>

<p style="text-align: center;"> Course: Natural Language Processing</p>

<p style="text-align: center;"> Group: 02 </p>

<p style="text-align: center;"> Phase 3 - Sentiment Analysis and Information Retrieval</p>

<p style="text-align: center;">  Presented by: Jeronimo Ramos Ruiz </p>

<p style="text-align: center;">  Tutor: ANDRES FELIPE HERNANDEZ GIRALDO</p>

<p style="text-align: center;"> UNIVERSIDAD NACIONAL ABIERTA Y A DISTANCIA - UNAD </p>

<p style="text-align: center;"> Medellín, October 2026</p>

---

# Sentiment Analysis for Mental Health — Lexicons, Semi-Supervised Labeling and Supervised Classification

Corpus: **Sentiment Analysis for Mental Health** (Kaggle, `Combined Data.csv`, referred to here as `mental_health.csv`).
Each row is a *statement* (a social-media post / short text) plus a *status* label
(Normal, Depression, Suicidal, Anxiety, Stress, Bipolar, Personality disorder).

| Stage | Content |
|---|---|
| 1 | Unsupervised, lexicon-based sentiment (VADER, plus a domain-adapted VADER and TextBlob for comparison) |
| 2 | Semi-supervised labeling: small labeled pool → propagate labels to the rest → validate on a gold test set |
| 3 | Supervised classifiers on the (propagated) corpus + metrics + interpretability (feature importance and LIME) |


## 0. Setup

In [ ]:
import importlib
import subprocess
import sys

librerias_proyecto = {"numpy": "numpy",
             "pandas": "pandas",
             "matplotlib":"matplotlib",
             "seaborn":"seaborn",
             "openpyxl": "openpyxl",
             "nltk": "nltk",
             "requests": "requests",
             "contractions": "contractions",
             "vaderSentiment": "vaderSentiment",
             "textblob": "textblob",
             "scikit-learn": "scikit-learn"
             }
             
print("====== INICIANDO VERIFICACIÓN DE ENTORNO ======\n")

# 2. Recorrer y validar cada librería de la lista
for nombre_importar, nombre_pip in librerias_proyecto.items():
    instalada = False
    
    while not instalada:
        try:
            # Intenta cargar la librería dinámicamente
            modulo = importlib.import_module(nombre_importar)
            instalada = True
            
            # Obtener la versión de forma segura
            version = getattr(modulo, "__version__", "Versión no expuesta")
            print(f"[{nombre_importar}] Lista para usar. Versión: {version}")
            
        except ImportError:
            print(f"[{nombre_importar}] No encontrada. Instalando vía pip: '{nombre_pip}'...")
            try:
                # Instala usando el entorno de Python que está ejecutando el script
                subprocess.check_call([sys.executable, "-m", "pip", "install", nombre_pip])
                print(f"[{nombre_importar}] Instalación enviada. Validando acceso...")
            except subprocess.CalledProcessError:
                print(f"[ERROR CRÍTICO] Falló la instalación de '{nombre_pip}'.")
                print("Revisa tu conexión a internet o los permisos de administrador.\n")
                break  # Rompe el ciclo while de esta librería para pasar a la siguiente

print("\n====== PROCESO DE CONFIGURACIÓN FINALIZADO ======")

In [ ]:
# pip install pandas numpy scikit-learn matplotlib vaderSentiment textblob lime
import re, warnings, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.semi_supervised import LabelSpreading
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             cohen_kappa_score)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
rng = np.random.RandomState(RANDOM_STATE)

DATA_PATH = "mental_health.csv"   # <-- path to your local copy (Kaggle: 'Combined Data.csv')
TARGET = "status"                  # label used as ground truth in Stages 2-3
N_LABELED_POOL = 3000              # size of the small randomly-selected subset (train + gold test)
GOLD_TEST_FRACTION = 0.30          # share of that subset reserved as the gold-standard test set

## 1. Loading and describing the corpus
Cleaning is deliberately minimal: drop an unnamed index column if present, drop empty statements and exact duplicate
statements (duplicates would otherwise leak between the labeled pool and the "unlabeled" rest).

In [ ]:
raw = pd.read_csv(DATA_PATH)
print("Raw shape:", raw.shape, "| columns:", list(raw.columns))

df = raw.loc[:, [c for c in raw.columns if not str(c).lower().startswith("unnamed")]].copy()
df.columns = [c.strip().lower() for c in df.columns]
assert {"statement", "status"} <= set(df.columns), "Expected columns 'statement' and 'status'"

n0 = len(df)
df = df.dropna(subset=["statement", "status"])
df["statement"] = df["statement"].astype(str).str.strip()
df = df[df["statement"].str.len() > 0]
n1 = len(df)
df = df.drop_duplicates(subset="statement").reset_index(drop=True)
print(f"Rows: {n0} raw -> {n1} after dropping empty -> {len(df)} after dropping duplicate statements")

df["n_words"] = df["statement"].str.split().str.len()
print("\nWords per statement:"); display(df["n_words"].describe().round(1).to_frame().T)

counts = df["status"].value_counts()
display(pd.DataFrame({"n": counts, "share_%": (100 * counts / len(df)).round(1)}))
ax = counts.plot.bar(figsize=(7, 3.5), color="#4C72B0", title="Class distribution (status)")
ax.set_ylabel("documents"); plt.tight_layout(); plt.show()

# Stage 1 — Unsupervised, lexicon-based sentiment analysis

### 1.1 Method choice and justification: VADER
**VADER** (Valence Aware Dictionary and sEntiment Reasoner) is the primary method, for these reasons:

* **Language/register fit.** It is built for **English** and tuned on **social-media text**, which matches the Reddit/Twitter-style
  origin of this corpus (informal wording, capitalisation, repeated punctuation, emoticons).
* **Rule-based handling that simpler lexicons lack.** Bing Liu (≈6.8k words) and MPQA are *flat* word lists: they count
  positive vs. negative words but have no intensity values, and no negation/intensifier logic. TextBlob (Pattern) averages
  adjective polarities and has only basic negation handling.
* **No training data needed**, fast, and interpretable (every score can be traced to words).

**How VADER works**
1. **Lexicon:** ≈7,500 tokens (incl. emoticons/slang) each with a mean human-rated valence from −4 (extremely negative) to +4 (extremely positive).
2. **Heuristics adjusting the valence:** *negation* (a negator in the 3 preceding tokens flips and dampens the score, ×−0.74),
   *intensifiers/dampeners* ("very", "extremely", "kind of"), *ALL-CAPS emphasis*, *exclamation/question marks*,
   and *"but"* (sentiment after "but" is up-weighted, before it down-weighted).
3. **Aggregation:** word valences are summed, then normalised to the **compound** score in [−1, +1] with
   `compound = x / sqrt(x² + 15)`; the neg/neu/pos proportions are also returned.
4. **Labels** (standard thresholds): compound ≥ +0.05 → *positive*; ≤ −0.05 → *negative*; otherwise *neutral*.

**Known limitation for this domain.** VADER's lexicon is general-purpose. Distress is often expressed without any
sentiment-bearing word ("I don't see the point anymore"), through sarcasm, or via clinical vocabulary absent from the lexicon.
Section 1.4 therefore also tests a small **domain-adapted** lexicon and compares it with TextBlob.

In [ ]:
analyzer = SentimentIntensityAnalyzer()

def vader_scores(texts, sia):
    return pd.DataFrame([sia.polarity_scores(t) for t in texts])

def to_label(compound, thr=0.05):
    return np.where(compound >= thr, "positive", np.where(compound <= -thr, "negative", "neutral"))

vs = vader_scores(df["statement"], analyzer)
df["vader_compound"] = vs["compound"].values
df["vader_label"] = to_label(df["vader_compound"].values)
df[["statement", "status", "vader_compound", "vader_label"]].head(5).assign(
    statement=lambda d: d["statement"].str.slice(0, 90) + "…")